In [1]:
import importlib
from journey_planner import JourneyPlanner

#importlib.reload(journey_planner)
"""
iceberg.kuci_iceberg
"""
schema = "iceberg.kuci_iceberg"

#schema = userns

jp = JourneyPlanner(schema=schema)

# prepare once
jp.prepare(
    regions=[
        "a7a21b73-6ffe-4fbf-a635-6e2b961f3072",  # Lausanne (district)
        "e168fd57-57a-407a-a350-0dcfbb55147f",  # Ouest lausannois
    ],
    rebuild=True
)


Connecting to iccluster129.iccluster.epfl.ch:8443...
Connected to Trino


/home/kuci/project/assignment-1/csa_data_handler.py:608: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, self.conn)


Creating stops...
build_stops: 0.60s
Creating stop_to_stop...
build_footpaths: 0.67s
Creating stop_times_seq...
build_stop_times_seq: 3.36s
Creating stop_times_trips_seq...
build_stop_times_trips_seq: 3.38s
Creating full_table_seq...
build_full_table_seq: 3.00s
Creating connections...
build_connections: 3.79s


/home/kuci/project/assignment-1/csa_data_handler.py:247: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, self.conn)


In [3]:
import time
import pickle
from pathlib import Path

test_cases = [
    # baseline all-stops
    {
        "name": "all_stops_mon_0600_walk0",
        "kwargs": dict(start_id=8501120, end_id=None, departs="06:00", day="monday", max_walk_m=0)
    },
    {
        "name": "all_stops_mon_0600_walk100",
        "kwargs": dict(start_id=8501120, end_id=None, departs="06:00", day="monday", max_walk_m=100)
    },
    {
        "name": "all_stops_mon_0600_walk300",
        "kwargs": dict(start_id=8501120, end_id=None, departs="06:00", day="monday", max_walk_m=300)
    },

    {
        "name": "all_stops_mon_0800_walk0",
        "kwargs": dict(start_id=8501120, end_id=None, departs="08:00", day="monday", max_walk_m=0)
    },
    {
        "name": "all_stops_mon_0800_walk100",
        "kwargs": dict(start_id=8501120, end_id=None, departs="08:00", day="monday", max_walk_m=100)
    },
    {
        "name": "all_stops_mon_0800_walk300",
        "kwargs": dict(start_id=8501120, end_id=None, departs="08:00", day="monday", max_walk_m=300)
    },

    {
        "name": "all_stops_mon_1230_walk0",
        "kwargs": dict(start_id=8501120, end_id=None, departs="12:30", day="monday", max_walk_m=0)
    },
    {
        "name": "all_stops_mon_1230_walk100",
        "kwargs": dict(start_id=8501120, end_id=None, departs="12:30", day="monday", max_walk_m=100)
    },
    {
        "name": "all_stops_mon_1230_walk300",
        "kwargs": dict(start_id=8501120, end_id=None, departs="12:30", day="monday", max_walk_m=300)
    },

    {
        "name": "all_stops_mon_1800_walk0",
        "kwargs": dict(start_id=8501120, end_id=None, departs="18:00", day="monday", max_walk_m=0)
    },
    {
        "name": "all_stops_mon_1800_walk100",
        "kwargs": dict(start_id=8501120, end_id=None, departs="18:00", day="monday", max_walk_m=100)
    },
    {
        "name": "all_stops_mon_1800_walk300",
        "kwargs": dict(start_id=8501120, end_id=None, departs="18:00", day="monday", max_walk_m=300)
    },

    {
        "name": "all_stops_mon_2200_walk100",
        "kwargs": dict(start_id=8501120, end_id=None, departs="22:00", day="monday", max_walk_m=100)
    },

    # weekday differences
    {
        "name": "all_stops_tue_1230_walk100",
        "kwargs": dict(start_id=8501120, end_id=None, departs="12:30", day="tuesday", max_walk_m=100)
    },
    {
        "name": "all_stops_wed_1230_walk100",
        "kwargs": dict(start_id=8501120, end_id=None, departs="12:30", day="wednesday", max_walk_m=100)
    },
    {
        "name": "all_stops_thu_1230_walk100",
        "kwargs": dict(start_id=8501120, end_id=None, departs="12:30", day="thursday", max_walk_m=100)
    },
    {
        "name": "all_stops_fri_1230_walk100",
        "kwargs": dict(start_id=8501120, end_id=None, departs="12:30", day="friday", max_walk_m=100)
    },
    {
        "name": "all_stops_sat_1230_walk100",
        "kwargs": dict(start_id=8501120, end_id=None, departs="12:30", day="saturday", max_walk_m=100)
    },
    {
        "name": "all_stops_sun_1230_walk100",
        "kwargs": dict(start_id=8501120, end_id=None, departs="12:30", day="sunday", max_walk_m=100)
    },

    # point-to-point
    {
        "name": "p2p_mon_0800_walk100",
        "kwargs": dict(start_id=8501120, end_id=8501117, departs="08:00", day="monday", max_walk_m=100)
    },
    {
        "name": "p2p_mon_1230_walk100",
        "kwargs": dict(start_id=8501120, end_id=8501117, departs="12:30", day="monday", max_walk_m=100)
    },
    {
        "name": "p2p_mon_1800_walk100",
        "kwargs": dict(start_id=8501120, end_id=8501117, departs="18:00", day="monday", max_walk_m=100)
    },
]

repeats = 50
baseline_dir = Path("route_result_baselines")
baseline_dir.mkdir(exist_ok=True)

print("=" * 80)
print("STEP 1: Functional result check against saved files")
print("=" * 80)

changed_cases = []
new_baselines = []

for case in test_cases:
    case_name = case["name"]
    baseline_file = baseline_dir / f"{case_name}.pkl"

    current_result = jp.route(**case["kwargs"])

    if not baseline_file.exists():
        with open(baseline_file, "wb") as f:
            pickle.dump(current_result, f)
        new_baselines.append(case_name)
        print(f"[BASELINE CREATED] {case_name}")
    else:
        with open(baseline_file, "rb") as f:
            old_result = pickle.load(f)

        if current_result == old_result:
           # print(f"[OK] {case_name}")
           bruh =0
        else:
            changed_cases.append(case_name)
            print(f"[CHANGED] {case_name}")

print()
print("=" * 80)
print("STEP 2: Timing benchmark")
print("=" * 80)

total_start = time.perf_counter()

for case in test_cases:
    for _ in range(repeats):
        _ = jp.route(**case["kwargs"])

total_end = time.perf_counter()

total_runs = len(test_cases) * repeats
elapsed = total_end - total_start

print(f"Ran {len(test_cases)} test cases x {repeats} repeats = {total_runs} routes")
print(f"Total routing time: {elapsed:.4f} seconds")
print(f"Average per route: {elapsed / total_runs:.6f} seconds")

print()
print("=" * 80)
print("STEP 3: Final summary")
print("=" * 80)

if new_baselines:
    print(f"Created {len(new_baselines)} new baseline file(s).")
    for name in new_baselines:
        print(f"  - {name}")

if changed_cases:
    print(f"{len(changed_cases)} case(s) changed compared to saved baseline:")
    for name in changed_cases:
        print(f"  - {name}")
else:
    if not new_baselines:
        print("No result changes detected compared to saved baseline files.")

STEP 1: Functional result check against saved files


TypeError: argument of type 'NoneType' is not iterable